# Create Prediction Inputs

Build the target and the readings from 1 and 24 hours earlier.

Open this notebook in `notebooks/M3/`. Run the cells in order. Saving again replaces this notebook's output files.


## Shared folder setup

In [1]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction')

## Read the hourly file

In [2]:
import pandas as pd
hourly = pd.read_csv(PROJECT_ROOT / "data/pilot/hourly_sample.csv")
hourly.head()

,LCLid,hour_start,kWh,n_valid,is_complete
0,MAC000414,2013-01-01 00:00:00+00:00,0.273,2,True
1,MAC000414,2013-01-01 01:00:00+00:00,0.193,2,True
2,MAC000414,2013-01-01 02:00:00+00:00,0.090,2,True
3,MAC000414,2013-01-01 03:00:00+00:00,0.091,2,True
4,MAC000414,2013-01-01 04:00:00+00:00,0.127,2,True


## Keep the three columns we need

In [3]:
hourly = hourly[["LCLid", "hour_start", "kWh"]]
hourly.columns = ["LCLid", "hour", "kwh"]
hourly.head()

,LCLid,hour,kwh
0,MAC000414,2013-01-01 00:00:00+00:00,0.273
1,MAC000414,2013-01-01 01:00:00+00:00,0.193
2,MAC000414,2013-01-01 02:00:00+00:00,0.090
3,MAC000414,2013-01-01 03:00:00+00:00,0.091
4,MAC000414,2013-01-01 04:00:00+00:00,0.127


## Read the hour as a date and time

In [4]:
hourly["hour"] = pd.to_datetime(hourly["hour"], utc=True)
hourly.head()

,LCLid,hour,kwh
0,MAC000414,2013-01-01 00:00:00+00:00,0.273
1,MAC000414,2013-01-01 01:00:00+00:00,0.193
2,MAC000414,2013-01-01 02:00:00+00:00,0.090
3,MAC000414,2013-01-01 03:00:00+00:00,0.091
4,MAC000414,2013-01-01 04:00:00+00:00,0.127


## List the expected hours

In [5]:
hours = pd.date_range(hourly["hour"].min(), hourly["hour"].max(), freq="h")
hours[:5]

DatetimeIndex(['2013-01-01 00:00:00+00:00', '2013-01-01 01:00:00+00:00',
               '2013-01-01 02:00:00+00:00', '2013-01-01 03:00:00+00:00',
               '2013-01-01 04:00:00+00:00'],
              dtype='datetime64[us, UTC]', freq='h')

## Make a household-and-hour grid

In [6]:
homes = sorted(hourly["LCLid"].unique())
grid = pd.MultiIndex.from_product([homes, hours], names=["LCLid", "target_hour"])
len(grid)

35040

## Index the observed values

In [7]:
values = hourly.set_index(["LCLid", "hour"])["kwh"]
values.index.names = ["LCLid", "target_hour"]
values.head()

LCLid      target_hour              
MAC000414  2013-01-01 00:00:00+00:00    0.273
           2013-01-01 01:00:00+00:00    0.193
           2013-01-01 02:00:00+00:00    0.090
           2013-01-01 03:00:00+00:00    0.091
           2013-01-01 04:00:00+00:00    0.127
Name: kwh, dtype: float64

## Include absent hours

In [8]:
values = values.reindex(grid)
values.head()

LCLid      target_hour              
MAC000414  2013-01-01 00:00:00+00:00    0.273
           2013-01-01 01:00:00+00:00    0.193
           2013-01-01 02:00:00+00:00    0.090
           2013-01-01 03:00:00+00:00    0.091
           2013-01-01 04:00:00+00:00    0.127
Name: kwh, dtype: float64

## Add the target

In [9]:
all_rows = values.reset_index(name="target_kwh")
all_rows.head()

,LCLid,target_hour,target_kwh
0,MAC000414,2013-01-01 00:00:00+00:00,0.273
1,MAC000414,2013-01-01 01:00:00+00:00,0.193
2,MAC000414,2013-01-01 02:00:00+00:00,0.090
3,MAC000414,2013-01-01 03:00:00+00:00,0.091
4,MAC000414,2013-01-01 04:00:00+00:00,0.127


## Find the previous hour

In [10]:
previous_time = all_rows["target_hour"] - pd.Timedelta(hours=1)
keys = pd.MultiIndex.from_arrays([all_rows["LCLid"], previous_time])
previous_values = values.reindex(keys)
all_rows["lag_1h_kwh"] = previous_values.values
all_rows.head()

,LCLid,target_hour,target_kwh,lag_1h_kwh
0,MAC000414,2013-01-01 00:00:00+00:00,0.273,NaN
1,MAC000414,2013-01-01 01:00:00+00:00,0.193,0.273
2,MAC000414,2013-01-01 02:00:00+00:00,0.090,0.193
3,MAC000414,2013-01-01 03:00:00+00:00,0.091,0.090
4,MAC000414,2013-01-01 04:00:00+00:00,0.127,0.091


## Find the same hour yesterday

In [11]:
yesterday_time = all_rows["target_hour"] - pd.Timedelta(hours=24)
keys = pd.MultiIndex.from_arrays([all_rows["LCLid"], yesterday_time])
yesterday_values = values.reindex(keys)
all_rows["lag_24h_kwh"] = yesterday_values.values
all_rows.iloc[22:27]

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh
22,MAC000414,2013-01-01 22:00:00+00:00,0.238,0.339,NaN
23,MAC000414,2013-01-01 23:00:00+00:00,0.090,0.238,NaN
24,MAC000414,2013-01-02 00:00:00+00:00,0.100,0.090,0.273
25,MAC000414,2013-01-02 01:00:00+00:00,0.121,0.100,0.193
26,MAC000414,2013-01-02 02:00:00+00:00,0.089,0.121,0.090


## Add calendar information

In [12]:
all_rows["hour"] = all_rows["target_hour"].dt.hour
all_rows["weekday"] = all_rows["target_hour"].dt.dayofweek
all_rows.iloc[22:27]

C:\Users\Mystery\AppData\Local\Temp\ipykernel_11820\3353378970.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  all_rows["weekday"] = all_rows["target_hour"].dt.dayofweek


,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday
22,MAC000414,2013-01-01 22:00:00+00:00,0.238,0.339,NaN,22,1
23,MAC000414,2013-01-01 23:00:00+00:00,0.090,0.238,NaN,23,1
24,MAC000414,2013-01-02 00:00:00+00:00,0.100,0.090,0.273,0,2
25,MAC000414,2013-01-02 01:00:00+00:00,0.121,0.100,0.193,1,2
26,MAC000414,2013-01-02 02:00:00+00:00,0.089,0.121,0.090,2,2


## Find rows with missing values

In [13]:
all_rows["row_status"] = "kept"
all_rows.loc[all_rows["lag_24h_kwh"].isna(), "row_status"] = "missing_lag_24h"
all_rows.loc[all_rows["lag_1h_kwh"].isna(), "row_status"] = "missing_lag_1h"
all_rows.loc[all_rows["target_kwh"].isna(), "row_status"] = "missing_target"
all_rows["row_status"].value_counts()

row_status
kept               34173
missing_lag_24h      422
missing_target       420
missing_lag_1h        25
Name: count, dtype: int64

## Keep the usable rows

In [14]:
features = all_rows[all_rows["row_status"] == "kept"]
features = features.drop(columns="row_status")
features.head()

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday
24,MAC000414,2013-01-02 00:00:00+00:00,0.100,0.090,0.273,0,2
25,MAC000414,2013-01-02 01:00:00+00:00,0.121,0.100,0.193,1,2
26,MAC000414,2013-01-02 02:00:00+00:00,0.089,0.121,0.090,2,2
27,MAC000414,2013-01-02 03:00:00+00:00,0.116,0.089,0.091,3,2
28,MAC000414,2013-01-02 04:00:00+00:00,0.108,0.116,0.127,4,2


## Count the rows

In [15]:
len(features)

34173

## Count missing values by home

In [16]:
counts = pd.crosstab(all_rows["LCLid"], all_rows["row_status"])
counts = counts.reindex(columns=["kept", "missing_target",
    "missing_lag_1h", "missing_lag_24h"], fill_value=0)
counts.columns = ["rows_kept", "excluded_missing_target",
    "excluded_missing_lag_1h", "excluded_missing_lag_24h"]
counts["grid_rows"] = all_rows.groupby("LCLid").size()
counts

,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,grid_rows
LCLid,,,,,
MAC000414,8534,100,8,118,8760
MAC000675,8727,4,3,26,8760
MAC001312,8735,1,1,23,8760
MAC005456,8177,315,13,255,8760


## Count missing fields separately

In [17]:
for field, name in [("target_kwh", "any_target_missing"),
                    ("lag_1h_kwh", "any_lag_1h_missing"), ("lag_24h_kwh", "any_lag_24h_missing")]:
    counts[name] = all_rows[field].isna().groupby(all_rows["LCLid"]).sum()
counts

,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,grid_rows,any_target_missing,any_lag_1h_missing,any_lag_24h_missing
LCLid,,,,,,,,
MAC000414,8534,100,8,118,8760,100,100,123
MAC000675,8727,4,3,26,8760,4,4,27
MAC001312,8735,1,1,23,8760,1,1,24
MAC005456,8177,315,13,255,8760,315,315,338


## Count observed targets on the first day

In [18]:
first_hour = all_rows.groupby("LCLid")["target_hour"].transform("min")
first_day = all_rows["target_hour"] < first_hour + pd.Timedelta(hours=24)
observed = first_day & all_rows["target_kwh"].notna()
counts["target_observed_in_first_24h"] = observed.groupby(all_rows["LCLid"]).sum()
counts

,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,grid_rows,any_target_missing,any_lag_1h_missing,any_lag_24h_missing,target_observed_in_first_24h
LCLid,,,,,,,,,
MAC000414,8534,100,8,118,8760,100,100,123,24
MAC000675,8727,4,3,26,8760,4,4,27,24
MAC001312,8735,1,1,23,8760,1,1,24,24
MAC005456,8177,315,13,255,8760,315,315,338,24


## Finish the count file

In [19]:
column_order = ["grid_rows", "rows_kept", "excluded_missing_target", "excluded_missing_lag_1h",
                "excluded_missing_lag_24h", "any_target_missing", "any_lag_1h_missing",
                "any_lag_24h_missing", "target_observed_in_first_24h"]
counts = counts[column_order].copy()
counts.loc["TOTAL"] = counts.sum()
counts = counts.reset_index()
counts

,LCLid,grid_rows,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,any_target_missing,any_lag_1h_missing,any_lag_24h_missing,target_observed_in_first_24h
0,MAC000414,8760,8534,100,8,118,100,100,123,24
1,MAC000675,8760,8727,4,3,26,4,4,27,24
2,MAC001312,8760,8735,1,1,23,1,1,24,24
3,MAC005456,8760,8177,315,13,255,315,315,338,24
4,TOTAL,35040,34173,420,25,422,420,420,512,96


## Create a new folder for this run

In [20]:
from pathlib import Path
output_dir = PROJECT_ROOT / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed')

## Save the features

In [21]:
feature_path = output_dir / "prediction_inputs_sample.csv"
features.to_csv(feature_path, index=False)
feature_path

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed/prediction_inputs_sample.csv')

## Save the counts

In [22]:
counts_path = output_dir / "prediction_input_counts_sample.csv"
counts.to_csv(counts_path, index=False)
counts_path

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed/prediction_input_counts_sample.csv')